# Activity 12: Logistic Regression with PyTorch

## Learning Objectives
By the end of this activity you will be able to:
- Subclass `nn.Module` to define a logistic regression model in PyTorch
- Write a complete manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`
- Use `DataLoader` to feed mini-batches during training
- Understand what `BCEWithLogitsLoss` is and why it is more numerically stable than `BCELoss + sigmoid`
- Plot training / validation loss curves and compare with sklearn baseline

---
> **Why PyTorch for logistic regression?**  
> PyTorch forces you to understand the training loop completely — no magic `.fit()`. This builds the deep intuition needed for custom architectures, research, and debugging.

## Part 1 — Imports & Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")

# --- Dataset ---
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=15,
    n_redundant=5, random_state=42
)
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_train_np, y_train_np, test_size=0.15, random_state=42
)

print(f"Train: {X_train_np.shape}  Val: {X_val_np.shape}  Test: {X_test_np.shape}")

## Part 2 — PyTorch Tensors & DataLoader

PyTorch does not work on NumPy arrays directly. We must convert to `torch.Tensor` and wrap in a `DataLoader` for mini-batch iteration.

In [ ]:
def to_tensor(X_np, y_np):
    """
    WHY float32?  PyTorch defaults to float32; mixing dtypes causes runtime errors.
    WHY unsqueeze(-1) on y?  nn.Linear output shape is (batch, 1);
                             y must match for BCEWithLogitsLoss.
    """
    X_t = torch.tensor(X_np, dtype=torch.float32)
    y_t = torch.tensor(y_np, dtype=torch.float32).unsqueeze(-1)  # (N,) → (N,1)
    return TensorDataset(X_t, y_t)

BATCH_SIZE = 32

train_ds = to_tensor(X_train_np, y_train_np)
val_ds   = to_tensor(X_val_np,   y_val_np)
test_ds  = to_tensor(X_test_np,  y_test_np)

# WHY shuffle=True for train only?  Shuffling breaks temporal / ordering bias.
# Val/test are not shuffled so metrics are stable across runs.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Batches per epoch: {len(train_loader)}")

## Part 3 — Model Definition

```
Input (20,)  →  nn.Linear(20, 1)  →  raw logit  →  BCEWithLogitsLoss applies sigmoid internally
```

**Why `BCEWithLogitsLoss` instead of `BCELoss`?**  
`BCEWithLogitsLoss` fuses sigmoid + binary cross-entropy in a numerically stable way using the log-sum-exp trick. Using `sigmoid → BCELoss` separately can cause gradients to vanish or explode.

In [ ]:
class LogisticRegressionPyTorch(nn.Module):
    """
    Logistic regression as a single linear layer.
    The sigmoid is NOT inside the model — BCEWithLogitsLoss handles it.
    COMMON ERROR: Adding sigmoid inside the model AND using BCEWithLogitsLoss
                  applies sigmoid twice → predictions stuck near 0.5.
    """
    def __init__(self, input_dim):
        super().__init__()
        # WHY bias=True?  The bias is the intercept — always needed unless
        # your data is already centred such that P(y=1) = 0.5 at origin.
        self.linear = nn.Linear(input_dim, 1, bias=True)

    def forward(self, x):
        return self.linear(x)   # returns raw logit — NOT probability

    def predict_proba(self, x):
        """Apply sigmoid to convert logit → probability for inference."""
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))

    def predict(self, x, threshold=0.5):
        proba = self.predict_proba(x)
        return (proba >= threshold).float()

# Instantiate
input_dim = X_train_np.shape[1]
model = LogisticRegressionPyTorch(input_dim).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}  ({input_dim} weights + 1 bias)")

## Part 4 — Training Loop

In [ ]:
criterion = nn.BCEWithLogitsLoss()
# WHY Adam?  Adaptive learning rates converge faster than vanilla SGD for tabular data.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

# --- Training utilities ---
def run_epoch(loader, model, criterion, optimizer, training):
    """Single epoch of training or validation."""
    model.train(training)   # sets BatchNorm / Dropout mode (none here, but good habit)
    total_loss, correct, total = 0.0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        if training:
            # WHY zero_grad FIRST?  PyTorch accumulates gradients by default.
            # Forgetting zero_grad is one of the most common PyTorch bugs.
            optimizer.zero_grad()

        logits = model(X_batch)
        loss   = criterion(logits, y_batch)

        if training:
            loss.backward()    # compute gradients
            optimizer.step()   # update weights

        # Accumulate metrics
        total_loss += loss.item() * len(X_batch)
        preds   = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y_batch).sum().item()
        total   += len(X_batch)

    return total_loss / total, correct / total


# --- Main training loop ---
N_EPOCHS   = 150
PATIENCE   = 15

train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []
best_val_loss = float('inf')
patience_counter = 0
best_weights = None

for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   model, criterion, optimizer, training=False)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)

    # --- Early stopping (manual) ---
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

# Restore best weights
model.load_state_dict(best_weights)
print(f"Best val loss : {best_val_loss:.4f}")

## Part 5 — Loss & Accuracy Curves

In [ ]:
epochs_range = range(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_range, train_losses, label='Train Loss',     color='steelblue')
axes[0].plot(epochs_range, val_losses,   label='Val Loss',       color='crimson', linestyle='--')
axes[0].set_title('BCEWithLogitsLoss')
axes[0].set_xlabel('Epoch');
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, train_accs, label='Train Accuracy', color='steelblue')
axes[1].plot(epochs_range, val_accs,   label='Val Accuracy',   color='crimson', linestyle='--')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("PyTorch Logistic Regression — Training History", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 6 — Test Evaluation & sklearn Comparison

In [ ]:
# Gather all test predictions
all_preds, all_labels = [], []
for X_batch, y_batch in test_loader:
    preds = model.predict(X_batch.to(device)).cpu().numpy().flatten()
    all_preds.extend(preds)
    all_labels.extend(y_batch.numpy().flatten())

y_pred_pt = np.array(all_preds, dtype=int)
acc_pt     = accuracy_score(all_labels, y_pred_pt)

# sklearn baseline
sk_model = LogisticRegression(max_iter=1000, random_state=42)
sk_model.fit(X_train_np, y_train_np)
acc_sk = accuracy_score(y_test_np, sk_model.predict(X_test_np))

print("=" * 46)
print(f"  PyTorch accuracy  : {acc_pt:.4f}")
print(f"  sklearn accuracy  : {acc_sk:.4f}")
print("=" * 46)
print("\n--- PyTorch Classification Report ---")
print(classification_report(all_labels, y_pred_pt))

## Part 7 — Inspect Learned Weights

In [ ]:
weights = model.linear.weight.detach().cpu().numpy().flatten()
bias    = model.linear.bias.detach().cpu().numpy()[0]

feature_names = [f"Feature {i+1}" for i in range(input_dim)]
importance = pd.DataFrame({'Feature': feature_names, 'Weight': weights})
importance = importance.reindex(importance['Weight'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 6))
colors = ['steelblue' if w > 0 else 'crimson' for w in importance['Weight']]
plt.barh(importance['Feature'], importance['Weight'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Weight  (Log-Odds Contribution)')
plt.title('Learned Feature Weights — PyTorch Logistic Regression')
plt.tight_layout()
plt.show()
print(f"Bias (intercept): {bias:.4f}")

## Summary

| Step | Code pattern |
|---|---|
| Define model | `class Model(nn.Module): ...` |
| Loss | `nn.BCEWithLogitsLoss()` |
| Optimiser | `torch.optim.Adam(model.parameters())` |
| Training step | `zero_grad → forward → loss → backward → step` |
| Early stopping | Manual patience counter, save `state_dict()` |
| Inference | `torch.sigmoid(model(x))` |

**Key takeaways:**
- Never forget `optimizer.zero_grad()` — accumulated gradients give wrong updates
- `BCEWithLogitsLoss` is more numerically stable than `sigmoid → BCELoss`
- `model.train()` / `model.eval()` modes matter even without Dropout — good habit

**Next:** Activity 13 — Evaluation Metrics Engine (from scratch + K-Fold)